# 04 · Export

把 `data/processed/village_to_nearest_library_{profile}.csv` 整理成終端使用者方便看的格式：

- `output/data/tainan_library_drive_time.csv` — 行車版 CSV
- `output/data/tainan_library_walk_time.csv` — 步行版 CSV
- `output/data/tainan_library_drive_time.xlsx` — Excel，三個 sheet：「行車時間」、「步行時間」、「圖書館清單」


In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUT_DIR = ROOT / "output" / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

libs = pd.read_csv(RAW_DIR / "tainan_libraries.csv")

def load(profile):
    fp = PROC_DIR / f"village_to_nearest_library_{profile}.csv"
    if not fp.exists():
        print(f"⚠️  {fp.name} not found")
        return None
    df = pd.read_csv(fp, dtype={"village_id": str})
    return df.sort_values(["district", "village_name"]).reset_index(drop=True)

driving = load("driving")
walking = load("walking")

# Individual CSVs (utf-8-sig so Excel reads Chinese correctly)
if driving is not None:
    p = OUT_DIR / "tainan_library_drive_time.csv"
    driving.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"✅ {p.name}: {len(driving)} rows")

if walking is not None:
    p = OUT_DIR / "tainan_library_walk_time.csv"
    walking.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"✅ {p.name}: {len(walking)} rows")

# Combined Excel
xlsx = OUT_DIR / "tainan_library_drive_time.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    if driving is not None:
        driving.to_excel(w, sheet_name="行車時間", index=False)
    if walking is not None:
        walking.to_excel(w, sheet_name="步行時間", index=False)
    libs.to_excel(w, sheet_name="圖書館清單", index=False)
print(f"✅ Excel: {xlsx.name}")

(driving if driving is not None else walking).head()